# Week 1 Day 2 Lab - Introducing UnderCover

## LLMs and Agents on a commercial task: insurance claims

A claims handler at UnderCover Travel Insurance has already decided each of the 25 claims in its queue. We send every claim to a few models, along with the customer's policy record and the policy wording. Each model replies with a decision of approve, decline or refer, a rationale and a payout. Then we compare the answers with the handlers' decisions and look at what each model cost.

At the end we give the same job to a LangChain agent that uses tools to look up the policy and record its decision.

## 4 parts:

PART 1: Examine the data

PART 2: Run with some models

PART 3: Optional: larger models and Jev

PART 4: An Agent via LangChain

In [ ]:
import json
import os
import time
from pathlib import Path
from typing import Literal

import pandas as pd
import plotly.express as px
from dotenv import load_dotenv
from IPython.display import Markdown, display
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from openai import OpenAI
from openrouter import OpenRouter
from pydantic import BaseModel, Field

## Connect to OpenRouter

The OpenRouter key comes from the `.env` file you made on day 1 in the `undercover` folder.

In [ ]:
load_dotenv(override=True)
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

if openrouter_api_key:
    print("OpenRouter API key found")
else:
    print("No OpenRouter API key found. Add OPENROUTER_API_KEY to the .env file")

openrouter = OpenAI(base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key, timeout=120, max_retries=5)

## Part 1: Load and examine the data

The seed data is in the repo's `data` folder. `get_sample_data` returns the claims and the handlers' decisions on them, keyed by claim id. The model is sent the claims, the policies and the policy wording.

By default we work with the queue of 25 claims. If you are using free models with a daily limit, set `TINY = True` to keep 5 of them.

In [ ]:
ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "data/claims.json").is_file())
DATA = ROOT / "data"


def load(name):
    """Read one of the JSON seed files."""
    return json.loads((DATA / name).read_text(encoding="utf-8"))


def get_sample_data(tiny=False, held_out=False):
    """Return a list of claims, and the handlers' decisions on them keyed by claim id.

    held_out=True gives the twenty-five held-out claims in place of the queue.
    tiny=True keeps the five claims marked for the tiny version of either set.
    """
    claims = load("evals/held_out_claims.json" if held_out else "claims.json")
    decisions = load("evals/held_out_decisions.json" if held_out else "evals/decisions.json")
    if tiny:
        chosen = {decision["claim_id"] for decision in decisions if decision["tiny"]}
        claims = [claim for claim in claims if claim["claim_id"] in chosen]
        decisions = [decision for decision in decisions if decision["tiny"]]
    labels = {decision["claim_id"]: decision for decision in decisions}
    return claims, labels

In [ ]:
TINY = False

claims, labels = get_sample_data(tiny=TINY)
policies = {policy["policy_id"]: policy for policy in load("policies.json")}
policy_wording = (DATA / "policy_wording.md").read_text(encoding="utf-8")

print(f"{len(claims)} claims, {len(labels)} labels, {len(policies)} policies")
print(f"The policy wording is {len(policy_wording.split()):,} words")

### Explore the data

It's always good to dig into the data before running experiments.

In [ ]:
rows = []
for claim in claims:
    label = labels[claim["claim_id"]]
    rows.append({
        "claim_id": claim["claim_id"],
        "summary": claim["summary"],
        "claim_type": claim["claim_type"],
        "amount_claimed": claim["amount_claimed"],
        "difficulty": label["difficulty"],
        "decision": label["decision"],
        "payout": label["payout"],
    })

data = pd.DataFrame(rows)
data

Claim CLM-1015 might be a familiar example...

In [ ]:
home_alone = [claim for claim in claims if claim["claim_id"] == "CLM-1015"][0]
decision = labels["CLM-1015"]

display(Markdown(f"""**{home_alone['summary']}**

{home_alone['story']}

**Handler:** {decision['decision']} ${decision['payout']:,.2f}. {decision['reason']}
"""))

Here's another one that might ring a bell...

In [ ]:
# Load the whole queue, in case TINY is True
queue, queue_labels = get_sample_data()
hangover = [claim for claim in queue if claim["claim_id"] == "CLM-1010"][0]
decision = queue_labels["CLM-1010"]

display(Markdown(f"""**{hangover['summary']}**

{hangover['story']}

**Handler:** {decision['decision']} ${decision['payout']:,.2f}. {decision['reason']}
"""))

This chart counts the claims of each type, split by the handler's decision.

In [ ]:
fig = px.histogram(data, x="claim_type", color="decision", title="Claims by type")
fig.show()

Here's a breakdown by difficulty. The policy wording is enough to decide the routine and hard claims. The manual claims depend on the claims manual or the customer's history.

In [ ]:
fig = px.histogram(
    data,
    x="difficulty",
    color="decision",
    category_orders={"difficulty": ["routine", "hard", "manual"]},
    title="Claims by difficulty",
)
fig.show()

## Part 2: running with some LLMs

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../../assets/info.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#118AB2;">Reminder: Structured Outputs</h2>
            <span style="color:#118AB2;">With Structured Outputs the model replies in JSON that follows a schema we give it. We describe the schema as a Pydantic class and pass it to <code>parse()</code> as the <code>response_format</code>. The reply comes back as an object of that class in <code>response.choices[0].message.parsed</code>.
            </span>
        </td>
    </tr>
</table>

Starting by defining a Pydantic class to describe the output so we can use Structured Outputs.

The rationale comes before the decision to ensure the model thinks through the decision before making it.

In [ ]:
class ClaimDecision(BaseModel):
    """A claims handler's decision on one claim."""

    rationale: str = Field(description="2-3 sentences to explain the decision along with the relevant section of the policy")
    decision: Literal["approve", "decline", "refer"]
    payout: float = Field(description="The amount to pay in US dollars, or 0 for a decline or a referral")

## The prompt

The system prompt has the instructions and the whole policy wording. The user prompt has the customer's policy record and the claim, both as JSON.

In [ ]:
INSTRUCTIONS = """You are a claims handler at UnderCover Travel Insurance.
Decide the claim you are given, using the policy wording below and the customer's policy record.

Choose one decision:
- approve: the wording covers the claim and the required documents are attached. Work out the payout using section 10.
- decline: the wording does not cover the claim.
- refer: a colleague needs to look at it, because a required document is missing, or the story and the documents disagree, or the wording does not settle the question.
"""

SYSTEM_PROMPT = f"""{INSTRUCTIONS}
{policy_wording}
"""


def make_user_prompt(claim):
    """The customer's policy record and the claim, as JSON."""
    policy = policies[claim["policy_id"]]
    return f"""# Policy record

{json.dumps(policy, indent=2)}

# Claim

{json.dumps(claim, indent=2)}
"""


def make_messages(claim):
    """Build the messages for one claim: the task and the wording, then the policy record and the claim."""
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": make_user_prompt(claim)},
    ]

### Calling the LLM

In [ ]:
def decide(claim, model, effort):
    """Ask one model to decide one claim."""
    start = time.time()
    response = openrouter.chat.completions.parse(
        model=model,
        messages=make_messages(claim),
        response_format=ClaimDecision,
        reasoning_effort=effort,
        extra_body={"provider": {"require_parameters": True}},
    )
    answer = response.choices[0].message.parsed
    return {
        "model": model,
        "effort": effort,
        "technique": "LLM",
        "claim_id": claim["claim_id"],
        "decision": answer.decision,
        "payout": answer.payout,
        "rationale": answer.rationale,
        "cost": response.usage.cost,
        "seconds": round(time.time() - start, 1),
    }

In [ ]:
first = decide(claims[0], "openai/gpt-6-luna", "low")
first

### Now running multiple models

In [ ]:
def is_right(answer, label):
    """True when the decision matches the handler's and the payout is within a dollar."""
    return answer["decision"] == label["decision"] and abs(answer["payout"] - label["payout"]) <= 1


def run(model, effort, claims, labels):
    """Run the model on the claims and compare against the labels."""
    results = []
    for claim in claims:
        answer = decide(claim, model, effort)
        label = labels[claim["claim_id"]]
        results.append({**answer, "difficulty": label["difficulty"], "right": is_right(answer, label)})
    right = sum(result["right"] for result in results)
    cost = sum(result["cost"] for result in results)
    seconds = sum(result["seconds"] for result in results)
    print(f"{model} at {effort}: {right} of {len(results)} right for ${cost:.4f} in {seconds:.0f} seconds")
    return results

In [ ]:
luna_low = run("openai/gpt-6-luna", "low", claims, labels)

In [ ]:
luna_medium = run("openai/gpt-6-luna", "medium", claims, labels)

In [ ]:
gemini = run("google/gemini-3.5-flash-lite", "low", claims, labels)

## Results by difficulty

This counts the claims each entry got right at each level of difficulty. Expect nearly all of the manual claims to be wrong for now.

In [ ]:
scored = pd.DataFrame(luna_low + luna_medium + gemini)
scored.pivot_table(index=["model", "effort"], columns="difficulty", values="right", aggfunc="sum", sort=False)

## Where a model got it wrong

For each claim a model got wrong, this shows its rationale, the handler's reason and what the claim is there to test. Here we look at Luna at `low`. Pass in another set of results to see its misses.

In [ ]:
def show_misses(results, labels):
    """Show each claim in a set of results that the model got wrong, with its rationale and the handler's reason."""
    for result in results:
        if not result["right"]:
            label = labels[result["claim_id"]]
            display(Markdown(f"""**{result['claim_id']}, {label['difficulty']}**: {result['decision']} ${result['payout']:,.2f}, where the handler said {label['decision']} ${label['payout']:,.2f}

- Model: {result['rationale']}
- Handler: {label['reason']}
- Tests: {label['tests']}
"""))


show_misses(luna_low, labels)

### What it cost

This shows how many claims each entry got right, what it cost and how long it took for the whole queue.

In [ ]:
scored.groupby(["model", "effort"], sort=False)[["right", "cost", "seconds"]].sum()

## Part 4: Using LangChain create_agent

Our approaches so far have used a single LLM call with Structured Outputs.

Now we'll use a different approach: an AI agent.

An LLM, with tools, in a loop, to achieve a goal!

We'll use LangChain's `create_agent` to run the loop for us.

We give it 2 tools. `get_policy` looks up the customer's policy record. `record_decision` saves the decision.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../../assets/info.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#118AB2;">Reminder: LangChain create_agent</h2>
            <span style="color:#118AB2;">1. Describe the LLM with a LangChain class like <code>ChatOpenAI</code><br/>
            2. Turn a function into a tool with <code>@tool</code><br/>
            3. <code>create_agent</code> builds the agent and <code>invoke()</code> runs its loop<br/><br/>
            Behind the scenes, a LangGraph graph is created; we'll find out more in Week 2.
            </span>
        </td>
    </tr>
</table>

In [ ]:
@tool
def get_policy(policy_id: str) -> str:
    """Look up a customer's policy record: the plan, the add-ons, the purchase date, the trip and the travelers."""
    policy = policies.get(policy_id)
    return json.dumps(policy, indent=2) if policy else f"There is no policy {policy_id}"


def make_record_decision(recorded):
    """Give each model its own decision store."""
    @tool
    def record_decision(claim_id: str, decision: Literal["approve", "decline", "refer"], payout: float, rationale: str) -> str:
        """Record the decision on a claim in the claims system.

        payout is the amount to pay in US dollars, or 0 for a decline or a referral.
        rationale is 2-3 sentences to explain the decision along with the relevant section of the policy.
        """
        recorded[claim_id] = {"rationale": rationale, "decision": decision, "payout": payout}
        return f"Recorded {decision} on {claim_id}"

    return record_decision


Each model uses LangChain's `ChatOpenAI`, pointed at OpenRouter. Its system prompt has the same instructions and policy wording as before, plus a line about the tools. Each model records decisions in its own store so parallel runs stay independent.


## My solution

Find paid models on OpenRouter that cost less than Luna. Run an agent for each model and a Luna baseline in parallel, then compare accuracy, cost and time.


In [ ]:
from datetime import datetime, timezone
from decimal import Decimal

import requests

url = "https://openrouter.ai/api/v1/models"
response = requests.get(url, params={"limit": 75, "sort": "newest"}, timeout=30)
response.raise_for_status()
latest_models = response.json()["data"]

luna_input_price = Decimal("0.10") / 1_000_000
luna_completion_price = Decimal("0.50") / 1_000_000
cheaper_models = []

for model in latest_models:
    if "batch" in model["id"].lower():
        continue
    pricing = model.get("pricing") or {}
    if pricing.get("prompt") is None or pricing.get("completion") is None:
        continue
    input_price = Decimal(pricing["prompt"])
    completion_price = Decimal(pricing["completion"])
    if 0 < input_price < luna_input_price and 0 < completion_price < luna_completion_price:
        cheaper_models.append(model)

print(f"Found {len(cheaper_models)} cheaper paid models among the latest {len(latest_models)}.")
for model in cheaper_models:
    added = datetime.fromtimestamp(model["created"], timezone.utc).date()
    print(f"Model ID: {model['id']}")
    print(f"Model Name: {model['name']}")
    print(f"Added to OpenRouter: {added}")
    print(f"Model Description: {model['description']}")
    print(f"Input: ${Decimal(model['pricing']['prompt']) * 1_000_000:.4f} / 1M tokens")
    print(f"Completion: ${Decimal(model['pricing']['completion']) * 1_000_000:.4f} / 1M tokens")
    print("-" * 40)


In [ ]:
AGENT_SYSTEM_PROMPT = f"""{INSTRUCTIONS}
Use your tools to look up the customer's policy record and to record your decision.

{policy_wording}
"""


def make_claim_prompt(claim):
    """The claim alone, as JSON. The agent looks up the policy record for itself."""
    return f"""# Claim

{json.dumps(claim, indent=2)}
"""


def make_claim_agent(model_id):
    """Create one agent and decision store per model."""
    recorded = {}
    reasoning = {"reasoning_effort": "low"} if model_id == "openai/gpt-6-luna" else {}
    model = ChatOpenAI(model=model_id, base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key, timeout=120, max_retries=5, **reasoning)
    agent = create_agent(model=model, tools=[get_policy, make_record_decision(recorded)], system_prompt=AGENT_SYSTEM_PROMPT)
    return agent, recorded


Run the whole queue for the two newest cheaper models, plus `openai/gpt-6-luna` at `low`. Up to three models run concurrently; each agent processes its claims in order. For cheaper models, `default` means no reasoning setting is sent, so OpenRouter and the provider determine the behavior. OpenRouter's reported cost is summed across the agent's model calls. A claim with no recorded decision counts as wrong. Failures identify the model and preserve the original exception.


In [ ]:
from concurrent.futures import ThreadPoolExecutor


def decide_with_agent(claim, agent, model_id, recorded):
    """Run one claim and check the model's decision against the handler's decision."""
    start = time.time()
    result = agent.invoke({"messages": [{"role": "user", "content": make_claim_prompt(claim)}]})
    answer = recorded.get(claim["claim_id"], {"decision": "nothing recorded", "payout": 0.0, "rationale": ""})
    label = labels[claim["claim_id"]]
    cost = sum(message.response_metadata["token_usage"]["cost"] for message in result["messages"] if message.type == "ai")
    return {
        "model": model_id,
        "effort": "low" if model_id == "openai/gpt-6-luna" else "default",
        "technique": "Agent",
        "claim_id": claim["claim_id"],
        **answer,
        "difficulty": label["difficulty"],
        "right": is_right(answer, label),
        "cost": cost,
        "seconds": round(time.time() - start, 1),
    }


def run_model(model_id):
    """Run one model's agent over the claims queue."""
    try:
        agent, recorded = make_claim_agent(model_id)
        results = [decide_with_agent(claim, agent, model_id, recorded) for claim in claims]
        print(f"{model_id}: {len(recorded)} of {len(claims)} claims have a recorded decision")
        return results
    except Exception as exc:
        raise RuntimeError(f"Agent experiment failed: {model_id}") from exc


model_ids = list(dict.fromkeys(["openai/gpt-6-luna", *[model["id"] for model in cheaper_models[:2]]]))
with ThreadPoolExecutor(max_workers=4) as executor:
    agent_experiments = list(executor.map(run_model, model_ids))

agent_results = [result for results in agent_experiments for result in results]


In [ ]:
pd.DataFrame(agent_results).groupby(["model", "effort", "technique"], sort=False)[["right", "cost", "seconds"]].sum()

In [ ]:
show_misses(agent_results, labels)

## Putting it all together

Here is every experiment in one table. Cost is for the whole queue and speed is the average time for one claim. Jev isn't included because it only makes the decision.

In [ ]:
experiments = [luna_low, *agent_experiments]

rows = []
for results in experiments:
    right = sum(result["right"] for result in results)
    cost = sum(result["cost"] for result in results)
    seconds = sum(result["seconds"] for result in results)
    rows.append({
        "LLM": results[0]["model"],
        "reasoning": results[0]["effort"],
        "technique": results[0]["technique"],
        "accuracy": f"{right} / {len(results)}",
        "cost": f"${cost:.4f}",
        "speed": f"{seconds / len(results):.1f} seconds a claim",
    })

pd.DataFrame(rows)


<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#EF476F;">Your assignment</h2>
            <span style="color:#EF476F;">This lab is about experimenting with different LLMs and Agents to understand the accuracy, cost and speed.<br/><br/>
            Based on our initial experiments, Luna (low) is able to perform well at a low cost. More expensive models can perform better.<br/><br/>
            Challenge: with a goal of at least 15 / 25 correct consistently, using the agent, what is the cheapest model you can find?<br/><br/>
            I'd love it if you would consider making a PR with your version of the lab in the <code>community_contributions</code> folder. There are instructions on making a PR at <a href="https://edwarddonner.com/pr">edwarddonner.com/pr</a>.
            </span>
        </td>
    </tr>
</table>